# IMPORTANT
**Run the next block of code for the notebook**
> If any error occurs with the block of code run the Requirements Section

# Detect Minino
The first step in the configuration process is to select the correct serial port to which the Minino device is connected.

<b>Identify the Port:</b> To check the available ports and determine the correct one, run the following code block:


In [2]:
import sys
sys.path.append("lib")

import ipywidgets as widgets
from IPython.display import display, clear_output

from main import Notebook, SerialConnection

notebook = Notebook()
serial_conn = SerialConnection()


scan_button = widgets.Button(description="Scan Ports")
detect_button = widgets.Button(description="Detect Minino", icon="arrow-right")
dropdown_ports = widgets.Dropdown(description='Ports:', layout=widgets.Layout(width='20%'))
detect_minino_output = widgets.Output(layout={'border': '1px solid black'})

def scan_ports(b):
    dropdown_ports.options = notebook.detect_serial_ports(0)

def detect_minino(btn):
    
    if not serial_conn.connect(port=dropdown_ports.value):
        with detect_minino_output:
            clear_output() 
            print("Error conecting the port")
        return
        
    data = serial_conn.send_command_string_with_response('h')
    
    if "minino" in data:
        with detect_minino_output:
            clear_output() 
            print(f"Minino Detected at {dropdown_ports.value}")
    else:
        with detect_minino_output:
            clear_output() 
            print("Minino no detected")
        
    serial_conn.disconnect()

scan_button.on_click(scan_ports)
detect_button.on_click(detect_minino)

display(
    widgets.Box([scan_button, dropdown_ports, detect_button]),
    detect_minino_output
)

Box(children=(Button(description='Scan Ports', style=ButtonStyle()), Dropdown(description='Ports:', layout=Lay…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

# Hands on Lab #5 Wifi Deauthentication
## Learning Objectives
By the end of this lab, participants will be able to:
1. **Explain** the function of **deauthentication frames** in Wi‑Fi communications.
2. **Demonstrate** how deauthentication can be used to forcibly disconnect wireless clients.
3. **Capture** and **analyze** reassociation frames, probe requests, or handshake attempts following a deauthentication event.
4. **Identify** vulnerabilities in unprotected Wi‑Fi management frames that make these attacks possible.
5. **Perform** a targeted deauthentication attack using Minino against a specific device.
6. **Document** observed attack effects and **recommend** detection or mitigation strategies.

## What is Wifi Deauthenticator?
A Wi-Fi deauthentication attack is a type of denial-of-service attack that forces a device to disconnect from a wireless network by sending fake deauthentication frames

## Launching at Deauth Menu
To start the Wifi Deauthentication, first the Minino need to be configure. On the next code of section, it launch on Minino the Deauth Menu where you can set the respective configuration

In [4]:
import time
import ipywidgets as widgets
from IPython.display import display, clear_output

connect_button = widgets.Button(description='Connect',button_style = 'success')
deauth_menu = widgets.Button(description='Go to Deauth Menu')
display_error = widgets.Output(layout={'border': '1px solid black'})
is_connected = False

def button_function(btn):
    global is_connected

    if not is_connected:
        if not serial_conn.connect(port=dropdown_ports.value):
            with display_error:
                clear_output()
                print("Error connecting the port")
            return

        time.sleep(1)
        connect_button.description = 'Disconnect'
        connect_button.button_style = 'danger'
        is_connected = True

    else: 
        serial_conn.disconnect()
        time.sleep(1)
        connect_button.description = 'Connect'
        connect_button.button_style = 'success'
        is_connected = False


def go_to_deauth_menu(btn):
    global is_connected
    if not is_connected:
        with display_error:
            clear_output()
            print("You must connect first.")
        return

    serial_conn.send_command_string("launch deauth")


connect_button.on_click(button_function)
deauth_menu.on_click(go_to_deauth_menu)

display(
    widgets.Box([connect_button, dropdown_ports]),
    deauth_menu,
    display_error
)

Box(children=(Button(button_style='success', description='Connect', style=ButtonStyle()), Dropdown(description…

Button(description='Go to Deauth Menu', style=ButtonStyle())

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

## Setting Up Minino
Once you have launched the Minino application, it will automatically scan for surrounding Wi-Fi Access Points (APs).
<div style="text-align: center;">
  <img src="static/scan_ap.jpeg" width="300" />
</div>

A menu will then appear, which you can navigate using the designated buttons.
<div style="text-align: center;">
  <img src="static/deauth_menu.jpeg" width="300" />
</div>

The menu options are:

1. **Scan:** scan the Wi-Fi AP around (This step is often redundant as the networks are usually already listed, but the option is there to refresh the scan).

3. **Select:** Use this option to choose the specific Wi-Fi AP you intend to attack.

4. **Attack:** Here you will select the type of attack to perform. You have three available types: Broadcast, Rogue AP, and Combined.

5. **Run:** Selecting this final option will execute the chosen attack.

# Quick Reference
- **Deauthentication Frame** – Wi‑Fi management frame used to terminate a connection between a client and an AP. Can be exploited to disconnect devices.
- **Disassociation Frame** – Similar to a deauthentication frame, used to break the association between a client and an AP without ending authentication state.
- **Management Frame** – Control packets in Wi‑Fi used for association, authentication, and maintaining connections; often unencrypted.
- **Probe Request** – Frame sent by a client to find nearby Wi‑Fi networks; can reveal preferred SSIDs.
- **Handshake** – Exchange of authentication and encryption keys between a client and AP when connecting.